In [1]:
%pip install datasets evaluate transformers accelerate peft bitsandbytes
%pip install sacrebleu

We load the dataset from Hugging Face.

In [2]:
from datasets import load_dataset

raw_datasets = load_dataset("xmj2002/Chinese_modern_classical")

print(raw_datasets)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['info', 'modern', 'classical'],
        num_rows: 972467
    })
})


Let's take a closer look at the features of the training set:

In [3]:
raw_datasets["train"].features

{'info': Value('string'),
 'modern': Value('string'),
 'classical': Value('string')}

Let us take a look at the translations of the first two sentences:

In [4]:
raw_datasets["train"][:2]["modern"]

['故意露出一些破绽，以引诱敌人深入我方，乘机切断他的后援和前应，最终陷他于死地。',
 '这就如《易经》 噬嗑 卦中说的，咬坚硬的腊肉而伤了牙齿一样，敌人为贪求不应得的利益，必招致后患。']

In [5]:
raw_datasets["train"][:2]["classical"]

['假之以便，唆之使前，断其援应，陷之死地。', '遇毒，位不当也。']

Now we load the pre-trained tokenizer for the NLLB model and apply it to the Simplified Chinese -> Traditional Chinese pair.
We use `zho_Hans` for Simplified Chinese and `zho_Hant` for Traditional Chinese.

In [ ]:
max_tok_length = 128 # Increased length for Chinese characters

from transformers import AutoTokenizer

checkpoint = "facebook/nllb-200-distilled-600M"
# from flores200_codes import flores_codes
src_code = "zho_Hans"
tgt_code = "zho_Hant"
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint, 
    padding=True, 
    pad_to_multiple_of=8, 
    src_lang=src_code, 
    tgt_lang=tgt_code, 
    truncation=True, 
    max_length=max_tok_length,
    )

We define the preprocessing function to map `modern` to source and `classical` to target.

In [7]:
def preprocess_function(sample):
    model_inputs = tokenizer(
        sample["modern"], 
        text_target = sample["classical"],
        )
    return model_inputs

Check the preprocessing:

In [8]:
sample = raw_datasets["train"].select(range(2))
model_input = preprocess_function({
    "modern": list(sample["modern"]),
    "classical": list(sample["classical"]),
})
print(model_input)

{'input_ids': [[256200, 166675, 249485, 252223, 249191, 76163, 250944, 3, 248079, 249120, 250101, 254770, 253110, 248624, 250590, 249507, 248956, 249279, 248079, 253755, 250857, 249999, 250559, 19763, 250475, 250569, 249249, 249389, 250670, 248079, 188158, 253587, 248968, 250079, 249900, 249242, 253935, 2], [256200, 29299, 249718, 249978, 3, 250911, 250557, 3, 248059, 3, 248059, 3, 249054, 250102, 248506, 248079, 255389, 253016, 252902, 248506, 254157, 251284, 249856, 252901, 249568, 252643, 3, 88193, 248079, 253110, 248624, 249685, 254883, 249961, 249215, 250670, 249652, 248506, 37572, 248079, 249494, 252150, 251153, 250475, 250896, 253935, 2]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'labels': [[3, 248059, 252034, 250005, 57987, 248079, 252978, 2500

In [9]:
for sample in model_input['input_ids']:
    print(tokenizer.convert_ids_to_tokens(sample))

['zho_Hans', '▁故', '意', '露', '出', '一些', '破', '<unk>', ',', '以', '引', '诱', '敌', '人', '深', '入', '我', '方', ',', '乘', '机', '切', '断', '他的', '后', '援', '和', '前', '应', ',', '最终', '陷', '他', '于', '死', '地', '。', '</s>']
['zho_Hans', '▁这', '就', '如', '<unk>', '易', '经', '<unk>', '▁', '<unk>', '▁', '<unk>', '中', '说', '的', ',', '咬', '坚', '硬', '的', '腊', '肉', '而', '伤', '了', '牙', '<unk>', '一样', ',', '敌', '人', '为', '贪', '求', '不', '应', '得', '的', '利益', ',', '必', '招', '致', '后', '患', '。', '</s>']


In [10]:
tokenizer.batch_decode(model_input['input_ids'])

['zho_Hans 故意露出一些破<unk>,以引诱敌人深入我方,乘机切断他的后援和前应,最终陷他于死地。</s>',
 'zho_Hans 这就如<unk>易经<unk> <unk> <unk>中说的,咬坚硬的腊肉而伤了牙<unk>一样,敌人为贪求不应得的利益,必招致后患。</s>']

Apply preprocessing to the dataset. Note: This dataset only has a 'train' split. We will split it into train/validation/test.

In [11]:
# Split the dataset since it only has 'train'
# We select 12000 samples: 10000 train, 1000 validation, 1000 test
shuffled_dataset = raw_datasets["train"].shuffle(seed=42).select(range(12000))

# Split into train (10000) and rest (2000)
train_testvalid = shuffled_dataset.train_test_split(test_size=2000, seed=42)

# Split rest (2000) into validation (1000) and test (1000)
test_valid = train_testvalid["test"].train_test_split(test_size=1000, seed=42)

from datasets import DatasetDict
raw_datasets = DatasetDict({
    'train': train_testvalid['train'],
    'valid': test_valid['train'],
    'test': test_valid['test']
})
print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['info', 'modern', 'classical'],
        num_rows: 10000
    })
    valid: Dataset({
        features: ['info', 'modern', 'classical'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['info', 'modern', 'classical'],
        num_rows: 1000
    })
})


In [12]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter by length:

In [13]:
tokenized_datasets = tokenized_datasets.filter(lambda x: len(x["input_ids"]) <= max_tok_length and len(x["labels"]) <= max_tok_length , desc=f"Discarding source and target sentences with more than {max_tok_length} tokens")

Discarding source and target sentences with more than 128 tokens:   0%|          | 0/10000 [00:00<?, ? example…

Discarding source and target sentences with more than 128 tokens:   0%|          | 0/1000 [00:00<?, ? examples…

Discarding source and target sentences with more than 128 tokens:   0%|          | 0/1000 [00:00<?, ? examples…

In [ ]:
# Check dataset sizes after filtering
print("Dataset sizes after filtering:")
print(f"Train: {len(tokenized_datasets['train'])} samples")
print(f"Valid: {len(tokenized_datasets['valid'])} samples")
print(f"Test: {len(tokenized_datasets['test'])} samples")

In [14]:
dic = {}
for sample in tokenized_datasets['train']:
    sample_length = len(sample['input_ids'])
    if sample_length not in dic:
        dic[sample_length] = 1
    else:
        dic[sample_length] += 1 

for i in range(1,max_tok_length+1):
    if i in dic:
        print(f"{i:>2} {dic[i]:>3}")

 4   9
 5  32
 6  36
 7  39
 8  44
 9  74
10 111
11 141
12 190
13 194
14 244
15 258
16 262
17 266
18 305
19 286
20 283
21 318
22 300
23 284
24 280
25 278
26 259
27 224
28 265
29 252
30 243
31 234
32 198
33 233
34 205
35 202
36 162
37 179
38 165
39 161
40 151
41 151
42 147
43 139
44 118
45 105
46 119
47 112
48  77
49 118
50  98
51  91
52  76
53  66
54  76
55  63
56  56
57  55
58  40
59  56
60  46
61  44
62  45
63  38
64  39
65  35
66  35
67  32
68  34
69  33
70  31
71  25
72  19
73  20
74  20
75  14
76  20
77  24
78  15
79  13
80  12
81  13
82  14
83   5
84  10
85  11
86  10
87  11
88  13
89  12
90  10
91  11
92   3
93   4
94   9
95  10
96  10
97   4
98   2
99   6
100   1
101   5
102   5
103   2
104   4
105   2
106   1
107   4
108   3
109   7
110   1
112   3
113   1
114   3
115   1
117   3
118   4
119   2
120   3
121   1
122   1
123   2
124   1
125   2
126   1
127   1
128   1


Load model with quantization:

In [15]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [16]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint,
    quantization_config=quantization_config
    )

In [17]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False, gradient_checkpointing_kwargs={'use_reentrant':False})

Setup LoRA:

In [18]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    task_type="SEQ_2_SEQ_LM",
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

In [19]:
lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()

trainable params: 3,538,944 || all params: 618,612,736 || trainable%: 0.5721


In [20]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, 
    model=lora_model, 
    pad_to_multiple_of=8
    )

## Evaluation

In [21]:
from evaluate import load

metric = load("sacrebleu")

In [22]:
import numpy as np
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace negative ids in the labels as we can't decode them.
    labels = [
        [tokenizer.pad_token_id if j < 0 else j for j in label]
        for label in labels
    ]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

## Training

In [23]:
from transformers import Seq2SeqTrainingArguments

batch_size = 32
model_name = checkpoint.split("/")[-1]
args = Seq2SeqTrainingArguments(
    f"{model_name}-finetuned-zh-modern-to-classical",
    eval_strategy = "epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=2,
    predict_with_generate=True,
)

In [24]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    lora_model,
    args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['valid'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-1421629073.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your prof

## Inference

In [ ]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    checkpoint,
)

print(generation_config)

In [ ]:
test_batch_size = 32
batch_tokenized_test = tokenized_datasets['test'].batch(test_batch_size)

In [ ]:
number_of_batches = len(batch_tokenized_test["modern"])
output_sequences = []
for i in range(number_of_batches):
    inputs = tokenizer(
        batch_tokenized_test["modern"][i], 
        max_length=max_tok_length, 
        truncation=True, 
        return_tensors="pt", 
        padding=True)
    with torch.no_grad():
        output_batch = lora_model.generate(
            generation_config=generation_config, 
            input_ids=inputs["input_ids"].cuda(), 
            attention_mask=inputs["attention_mask"].cuda(), 
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_code), 
            max_length = max_tok_length, 
            num_beams=1, 
            do_sample=False,)
    output_sequences.extend(output_batch.cpu())

In [ ]:
result = compute_metrics((output_sequences,tokenized_datasets['test']["labels"]))
print(f'BLEU score: {result["bleu"]}')